# HAM10000 ResNet Fine-tuning
This notebook demonstrates how to fine-tune a ResNet model on the HAM10000 dataset using the `ResNetFineTuneTrainer` class.

In [1]:
import sys
import os

# Add the project root to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from dualbranch.ham10000_dataset_manager import HAM10000DatasetManager
from dualbranch.ResNetFinetuneTrain import ResNetFineTuneTrainer

In [2]:
# 1. Prepare the Data
# Initialize the dataset manager
data_dir = r"C:\Users\z004ps0b\Documents\dermascan\data"
manager = HAM10000DatasetManager(data_dir=data_dir)

# Load metadata (filtering for Melanoma vs Nevi as per trainer restrictions)
manager.load_cleaned_metadata(filter_melanoma_only=True)

# Split the data into train (70%), validation (15%), and test (15%) sets
train_df, val_df, test_df = manager.train_test_val_split(val_size=0.15, test_size=0.15)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

Training samples: 2766
Validation samples: 593
Test samples: 593


In [3]:
# 2. Configure the Trainer
trainer = ResNetFineTuneTrainer()

# Set up the model (ResNet50, 2 classes: nv vs mel)
trainer.set_model(model_name='resnet50', pretrained=True)

# Set data loaders
trainer.set_data_loaders(train_df, val_df, test_df)

# Configure optimizer and scheduler
trainer.configure_optimizer(optimizer_type='adam', lr=1e-4, weight_decay=1e-4)
trainer.configure_scheduler(scheduler_type='reduce_on_plateau', patience=3)

# Set output paths for artifacts
trainer.set_output_paths(
    model_save_dir="../results/models",
    results_csv_path="../results/training_results.csv",
    history_plot_dir="../results/plots"
)

c:\Users\z004ps0b\Documents\dermascan\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\z004ps0b\Documents\dermascan\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
# 3. Train the Model
# Train for a specified number of epochs
trainer.train(epochs=15)

Epoch 1/15
----------
train Loss: 0.4577 Acc: 0.8261
val Loss: 0.4077 Acc: 0.8280
Epoch 2/15
----------
train Loss: 0.3995 Acc: 0.8297
val Loss: 0.3699 Acc: 0.8381
Epoch 3/15
----------
train Loss: 0.3733 Acc: 0.8333
val Loss: 0.3452 Acc: 0.8364
Epoch 4/15
----------
train Loss: 0.3515 Acc: 0.8420
val Loss: 0.3286 Acc: 0.8381
Epoch 5/15
----------
train Loss: 0.3450 Acc: 0.8478
val Loss: 0.3204 Acc: 0.8449
Epoch 6/15
----------
train Loss: 0.3348 Acc: 0.8525
val Loss: 0.3134 Acc: 0.8516
Epoch 7/15
----------
train Loss: 0.3259 Acc: 0.8601
val Loss: 0.3086 Acc: 0.8499
Epoch 8/15
----------
train Loss: 0.3237 Acc: 0.8594
val Loss: 0.3028 Acc: 0.8600
Epoch 9/15
----------
train Loss: 0.3184 Acc: 0.8612
val Loss: 0.2963 Acc: 0.8634
Epoch 10/15
----------
train Loss: 0.3161 Acc: 0.8619
val Loss: 0.3075 Acc: 0.8415
Epoch 11/15
----------
train Loss: 0.3122 Acc: 0.8630
val Loss: 0.2945 Acc: 0.8634
Epoch 12/15
----------
train Loss: 0.3116 Acc: 0.8666
val Loss: 0.2946 Acc: 0.8803
Epoch 13/15
-

In [5]:
# 4. Evaluate and Save Results
trainer.evaluate()
trainer.save_results_to_csv()

Evaluation results - F1 (macro): 0.7290, F1 (weighted): 0.8534, AUC: 0.8599, Accuracy: 0.8617, Sensitivity: 0.4706, Specificity: 0.9430
